# Test In-Silico Results

In [38]:
import os
from collections import defaultdict, Counter

def parse_sequence_file(file_path):
    _, file_extension = os.path.splitext(file_path)

    if file_extension.lower() not in ['.fasta', '.fa', '.txt']:
        raise ValueError(f"Unsupported file extension: {file_extension}")

    def sequence_generator():
        current_id = None
        current_seq = []
        with open(file_path, 'r') as file:
            for line in file:
                line = line.strip()
                if not line:  # Skip empty lines
                    continue
                if line.startswith('>'):
                    if current_id:
                        seq = ''.join(current_seq).strip()
                        if seq:
                            current_id = current_id.lstrip('>')
                            header_parts = current_id.split(" ", 1)
                            seqid = header_parts[0]
                            taxonomy = header_parts[1].strip() if len(header_parts) > 1 and header_parts[1].strip() else None

                            yield seqid, taxonomy, seq
                    current_id = line[1:] if file_extension.lower() in ['.fasta', '.fa'] else ';'.join(line[1:].split(';')[:2])
                    current_seq = []
                else:
                    current_seq.append(line)
        if current_id:
            seq = ''.join(current_seq).strip()
            if seq:
                current_id = current_id.lstrip('>')
                header_parts = current_id.split(";", 1)
                seqid = header_parts[0]
                taxonomy = header_parts[1].strip() if len(header_parts) > 1 and header_parts[1].strip() else None

                yield seqid, taxonomy, seq

    return sequence_generator()

def process_sequences(generator):
    total_sequences = 0
    unique_seqids = set()
    unique_taxonomies = set()
    unique_sequences = set()
    id_to_seq = {}
    seq_lengths = []

    for seqid, taxonomy, seq in generator:
        # print(f"{seqid} - {taxonomy} - {seq}")
        total_sequences += 1
        unique_seqids.add(seqid)
        unique_taxonomies.add(taxonomy)
        unique_sequences.add(seq)
        id_to_seq[seqid] = (taxonomy, seq)
        seq_lengths.append(len(seq))

    return {
        'total': total_sequences,
        'unique_seqids': unique_seqids,
        'unique_taxonomies': unique_taxonomies,
        'unique_sequences': unique_sequences,
        'id_to_seq': id_to_seq,
        'seq_lengths': seq_lengths
    }

def check_intersection_between_results(folderA, folderB):
    filesA = [f for f in os.listdir(folderA) if f.endswith(('.fasta', '.fa', '.txt'))]
    filesB = [f for f in os.listdir(folderB) if f.endswith(('.fasta', '.fa', '.txt'))]

    base_filesA = {os.path.splitext(f)[0]: f for f in filesA}
    base_filesB = {os.path.splitext(f)[0]: f for f in filesB}

    files_same_base_name = set(base_filesA.keys()) & set(base_filesB.keys())

    if not files_same_base_name:
        print("No matching files found between the two folders. Check folder direcotry and content before re-running.")
        return

    results = defaultdict(dict)

    for file_name in files_same_base_name:
        fileA = base_filesA[file_name]
        fileB = base_filesB[file_name]
        pathA = os.path.join(folderA, fileA)
        pathB = os.path.join(folderB, fileB)

        try:
            # print(f"Processing {fileA} from Folder A...")
            recordsA = parse_sequence_file(pathA)
            resultsA = process_sequences(recordsA)
            seqidsA = resultsA['unique_seqids']
            taxonomiesA = resultsA['unique_taxonomies']
            sequencesA = resultsA['unique_sequences']

            # print(f"Processing {fileB} from Folder B...")
            recordsB = parse_sequence_file(pathB)
            resultsB = process_sequences(recordsB)
            seqidsB = resultsB['unique_seqids']
            taxonomiesB = resultsB['unique_taxonomies']
            sequencesB = resultsB['unique_sequences']
        except Exception as e:
            print(f"Error reading files: {fileA} and {fileB}, Error: {e}")
            continue

        repeated_headersA = resultsA['total'] - len(seqidsA)
        repeated_headersB = resultsB['total'] - len(seqidsB)
        repeated_sequencesA = resultsA['total'] - len(sequencesA)
        repeated_sequencesB = resultsB['total'] - len(sequencesB)

        sequence_count_diff = resultsB['total'] - resultsA['total']

        avg_lengthA = sum(resultsA['seq_lengths']) / len(resultsA['seq_lengths']) if resultsA['seq_lengths'] else 0
        avg_lengthB = sum(resultsB['seq_lengths']) / len(resultsB['seq_lengths']) if resultsB['seq_lengths'] else 0

        min_lengthA = min(resultsA['seq_lengths']) if resultsA['seq_lengths'] else 0
        max_lengthA = max(resultsA['seq_lengths']) if resultsA['seq_lengths'] else 0
        min_lengthB = min(resultsB['seq_lengths']) if resultsB['seq_lengths'] else 0
        max_lengthB = max(resultsB['seq_lengths']) if resultsB['seq_lengths'] else 0

        avg_diff = round(abs(avg_lengthA - avg_lengthB), 2)

        intersection_seqids = seqidsA & seqidsB
        only_seqids_in_A = seqidsA - seqidsB
        only_seqids_in_B = seqidsB - seqidsA

        intersection_taxonomies = taxonomiesA & taxonomiesB
        only_taxonomies_in_A = taxonomiesA - taxonomiesB
        only_taxonomies_in_B = taxonomiesB - taxonomiesA

        all_sequences_in_A = sequencesB.issubset(sequencesA)
        all_seqids_in_A = seqidsB.issubset(seqidsA)
        all_taxonomies_in_A = taxonomiesB.issubset(taxonomiesA)

        results[file_name] = {
            "total_A": resultsA['total'],
            "avg_lenA": round(avg_lengthA, 2),
            "min_lenA": min_lengthA,
            "max_lenA": max_lengthA,
            "total_B": resultsB['total'],
            "avg_lenB": round(avg_lengthB, 2),
            "min_lenB": min_lengthB,
            "max_lenB": max_lengthB,
            "intersection_seqids": len(intersection_seqids),
            "only_seqids_in_A": len(only_seqids_in_A),
            "only_seqids_in_B": len(only_seqids_in_B),
            "intersection_taxonomies": len(intersection_taxonomies),
            "only_taxonomies_in_A": len(only_taxonomies_in_A),
            "only_taxonomies_in_B": len(only_taxonomies_in_B),
            "total_unique_taxonomies_A": len(resultsA['unique_taxonomies']),
            "total_unique_taxonomies_B": len(resultsB['unique_taxonomies']),
            "avg_diff": avg_diff,
            "repeated_headersA": repeated_headersA,
            "repeated_headersB": repeated_headersB,
            "repeated_sequencesA": repeated_sequencesA,
            "repeated_sequencesB": repeated_sequencesB,
            "sequence_count_diff": sequence_count_diff,
            "seqids_only_in_A": list(only_seqids_in_A),
            "taxonomies_only_in_A": list(only_taxonomies_in_A)
        }

        print(f"Comparison for {file_name}:")
        print(f"  Total sequences in {file_name} from Folder A: {results[file_name]['total_A']}")
        print(f"  Total sequences in {file_name} from Folder B: {results[file_name]['total_B']}")
        print(f"  Difference in sequence count (B - A): {results[file_name]['sequence_count_diff']}")
        print(f"  Intersection of seqIDs (Folder A and B): {results[file_name]['intersection_seqids']}")
        print(f"  Intersection of taxonomies (Folder A and B): {results[file_name]['intersection_taxonomies']}")
        print(f"  Minimum Length of sequences in Folder A: {results[file_name]['min_lenA']}")
        print(f"  Maximum Length of sequences in Folder A: {results[file_name]['max_lenA']}")
        print(f"  Minimum Length of sequences in Folder B: {results[file_name]['min_lenB']}")
        print(f"  Maximum Length of sequences in Folder B: {results[file_name]['max_lenB']}")
        print(f"  Repeated headers in File A: {results[file_name]['repeated_headersA']}")
        print(f"  Repeated headers in File B: {results[file_name]['repeated_headersB']}")
        print(f"  Repeated sequences in File A: {results[file_name]['repeated_sequencesA']}")
        print(f"  Repeated sequences in File B: {results[file_name]['repeated_sequencesB']}")
        print(f"  Number of seqIDs only in File A: {results[file_name]['only_seqids_in_A']}")
        print(f"  Number of seqIDs only in File B: {results[file_name]['only_seqids_in_B']}")
        print(f"  Number of taxonomies only in File A: {results[file_name]['only_taxonomies_in_A']}")
        print(f"  Number of taxonomies only in File B: {results[file_name]['only_taxonomies_in_B']}")
        print(f"  Total unique taxonomies in A: {results[file_name]['total_unique_taxonomies_A']}")
        print(f"  Total unique taxonomies in B: {results[file_name]['total_unique_taxonomies_B']}")

        if all_sequences_in_A:
            print("All sequences from B are present in A.")
        else:
            print("Not all sequences from B are present in A.")

        if all_seqids_in_A:
            print("All seqIDs from B are present in A.")
        else:
            print("Not all seqIDs from B are present in A.")

        if all_taxonomies_in_A:
            print("All taxonomies from B are present in A.")
        else:
            print("Not all taxonomies from B are present in A.")

        print("-" * 60)

## Cutadapt 3 & 5

### New Script
Same sequence IDs are present in both but sequences are different as different number of allowed mismatches will haev different scores.

In [2]:
folder_cutadapt5 = "../../data/output_data/test-filter-relaxed/all_barcodes_w_pbr"
folder_cutadapt3 = "../../data/output_data/test-filter-relaxed/insert"

check_intersection_between_results(folder_cutadapt3, folder_cutadapt5)

FileNotFoundError: [Errno 2] No such file or directory: '../../data/output_data/test-filter-relaxed/insert'

## Cutadapt & PGA

### New Script

### Diat.barcode
Percid: 75
Cov: 99
All hits vs top hits

In [39]:
folder_pga_all_hits = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/diat-barcode/inserts/id75-cov99-allhits/pga/filtered"
folder_pga_best_hits = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/diat-barcode/inserts/id75-cov99-besthits/pga/filtered"
folder_pga_20_best_hits = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/diat-barcode/inserts/id75-cov99-20besthits/pga/filtered"

check_intersection_between_results(folder_pga_best_hits, folder_pga_all_hits)

Comparison for rbcL_708F1_R3-2:
  Total sequences in rbcL_708F1_R3-2 from Folder A: 5374
  Total sequences in rbcL_708F1_R3-2 from Folder B: 2883287
  Difference in sequence count (B - A): 2877913
  Intersection of seqIDs (Folder A and B): 5374
  Intersection of taxonomies (Folder A and B): 1262
  Minimum Length of sequences in Folder A: 178
  Maximum Length of sequences in Folder A: 266
  Minimum Length of sequences in Folder B: 178
  Maximum Length of sequences in Folder B: 266
  Repeated headers in File A: 0
  Repeated headers in File B: 2877902
  Repeated sequences in File A: 2978
  Repeated sequences in File B: 2880034
  Number of seqIDs only in File A: 0
  Number of seqIDs only in File B: 11
  Number of taxonomies only in File A: 0
  Number of taxonomies only in File B: 0
  Total unique taxonomies in A: 1262
  Total unique taxonomies in B: 1262
Not all sequences from B are present in A.
Not all seqIDs from B are present in A.
All taxonomies from B are present in A.
--------------

In [35]:
folder_pga_all_hits_12S = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/12S/id75-cov99-allhits/pga/filtered"
folder_pga_best_hits_12S = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/12S/id75-cov99-besthit/pga/filtered"
folder_pga_20_best_hits_12s = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/12S/id75-cov99-20besthits/pga/filtered"

check_intersection_between_results(folder_pga_best_hits_12S, folder_pga_all_hits_12S)

Comparison for 12S_Am12s:
  Total sequences in 12S_Am12s from Folder A: 741
  Total sequences in 12S_Am12s from Folder B: 2852
  Difference in sequence count (B - A): 2111
  Intersection of seqIDs (Folder A and B): 741
  Intersection of taxonomies (Folder A and B): 1
  Minimum Length of sequences in Folder A: 250
  Maximum Length of sequences in Folder A: 260
  Minimum Length of sequences in Folder B: 250
  Maximum Length of sequences in Folder B: 263
  Repeated headers in File A: 0
  Repeated headers in File B: 2109
  Repeated sequences in File A: 465
  Repeated sequences in File B: 2554
  Number of seqIDs only in File A: 0
  Number of seqIDs only in File B: 2
  Number of taxonomies only in File A: 0
  Number of taxonomies only in File B: 0
  Total unique taxonomies in A: 1
  Total unique taxonomies in B: 1
Not all sequences from B are present in A.
Not all seqIDs from B are present in A.
All taxonomies from B are present in A.
---------------------------------------------------------

In [36]:
taxa_pga_all_hits_12S = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/12S/taxid/id75-cov99-allhits/pga/filtered"
taxa_pga_best_hits_12S = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/12S/taxid/id75-cov95-besthit/pga/filtered"

check_intersection_between_results(taxa_pga_best_hits_12S, taxa_pga_all_hits_12S)

Comparison for 12S_Am12s:
  Total sequences in 12S_Am12s from Folder A: 741
  Total sequences in 12S_Am12s from Folder B: 2852
  Difference in sequence count (B - A): 2111
  Intersection of seqIDs (Folder A and B): 741
  Intersection of taxonomies (Folder A and B): 212
  Minimum Length of sequences in Folder A: 250
  Maximum Length of sequences in Folder A: 260
  Minimum Length of sequences in Folder B: 250
  Maximum Length of sequences in Folder B: 263
  Repeated headers in File A: 0
  Repeated headers in File B: 2109
  Repeated sequences in File A: 465
  Repeated sequences in File B: 2554
  Number of seqIDs only in File A: 0
  Number of seqIDs only in File B: 2
  Number of taxonomies only in File A: 0
  Number of taxonomies only in File B: 0
  Total unique taxonomies in A: 212
  Total unique taxonomies in B: 212
Not all sequences from B are present in A.
Not all seqIDs from B are present in A.
All taxonomies from B are present in A.
---------------------------------------------------

In [9]:
folder_pga_best_hits_12S = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/12S/id75-cov99-besthit/pga/filtered"
folder_cutadapt5 = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/12S/cutadapt5/all_barcodes_w_pbr/filtered"

check_intersection_between_results(folder_cutadapt5, folder_pga_best_hits_12S)

Comparison for 12S_Hauck2019a:
  Total sequences in 12S_Hauck2019a from Folder A: 762
  Total sequences in 12S_Hauck2019a from Folder B: 614
  Difference in sequence count (B - A): -148
  Intersection (Folder A and B): 608
  Minimum Length of sequences in Folder A: 198
  Maximum Length of sequences in Folder A: 212
  Minimum Length of sequences in Folder B: 198
  Maximum Length of sequences in Folder B: 212
  Repeated headers in File A: 0
  Repeated headers in File B: 0
  Repeated sequences in File A: 509
  Repeated sequences in File B: 392
  Number of headers only in File A: 154
  Number of headers only in File B: 6
Not all sequences from A are present in B.
Not all headers from A are present in B.
------------------------------------------------------------
Comparison for 12S_MarVer1:
  Total sequences in 12S_MarVer1 from Folder A: 736
  Total sequences in 12S_MarVer1 from Folder B: 756
  Difference in sequence count (B - A): 20
  Intersection (Folder A and B): 736
  Minimum Length o

In [10]:
folder_pga_best_hits_diat = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/diat-barcode/inserts/id75-cov99-besthits/pga/filtered"
folder_cutadapt5_diat= "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/diat-barcode/cutadapt5/all_barcodes_w_pbr/filtered"

check_intersection_between_results(folder_cutadapt5_diat, folder_pga_best_hits_diat)

Comparison for rbcL_708F1_R3-1:
  Total sequences in rbcL_708F1_R3-1 from Folder A: 5054
  Total sequences in rbcL_708F1_R3-1 from Folder B: 5374
  Difference in sequence count (B - A): 320
  Intersection (Folder A and B): 4773
  Minimum Length of sequences in Folder A: 178
  Maximum Length of sequences in Folder A: 263
  Minimum Length of sequences in Folder B: 178
  Maximum Length of sequences in Folder B: 266
  Repeated headers in File A: 0
  Repeated headers in File B: 0
  Repeated sequences in File A: 2832
  Repeated sequences in File B: 2978
  Number of headers only in File A: 281
  Number of headers only in File B: 601
All sequences from A are present in B.
Not all headers from A are present in B.
------------------------------------------------------------
Comparison for rbcL_708F-DEG_R3-DEG:
  Total sequences in rbcL_708F-DEG_R3-DEG from Folder A: 5067
  Total sequences in rbcL_708F-DEG_R3-DEG from Folder B: 5374
  Difference in sequence count (B - A): 307
  Intersection (Fold

## Side Transformations

### Fix Paths for Previous Version

In [ ]:
import os

def rename_file_extensions(folder, old_extension, new_extension):
    for filename in os.listdir(folder):
        if filename.endswith(old_extension):
            base_name = os.path.splitext(filename)[0]
            new_filename = f"{base_name}{new_extension}"

            old_file = os.path.join(folder, filename)
            new_file = os.path.join(folder, new_filename)

            os.rename(old_file, new_file)
            print(f"Renamed: {old_file} -> {new_file}")

folder1 = "/home/camilababo/Documents/DNAquaIMG/joana_scripts/Trial/cutadapt/12S/test_results/cutadapt3"
folder2 = "/home/camilababo/Documents/DNAquaIMG/joana_scripts/Trial/cutadapt/12S/test_results/cutadapt5"
old_extension1 = ".cutadapt"
old_extension2 = ".maxerror"
new_extension = ".txt"

rename_file_extensions(folder1, old_extension1, new_extension)
rename_file_extensions(folder2, old_extension2, new_extension)

In [ ]:
folder3 = "/home/camilababo/Documents/DNAquaIMG/joana_scripts/Trialfolder_pga_fasta/cutadapt/12S/pga"
rename_file_extensions(folder3, ".pga", ".txt")

In [ ]:
folder1 = "/home/camilababo/Documents/DNAquaIMG/joana_scripts/Trial/cutadapt/12S/cutadapt3"
folder2 = "/home/camilababo/Documents/DNAquaIMG/joana_scripts/Trial/cutadapt/12S/cutadapt5"
folder3 = "/home/camilababo/Documents/DNAquaIMG/joana_scripts/Trial/cutadapt/12S/pga"
old_extension1 = ".cutadapt"
old_extension2 = ".maxerror"
old_extension3 = ".pga"
new_extension = ".txt"

#rename_file_extensions(folder1, old_extension1, new_extension)
rename_file_extensions(folder2, old_extension2, new_extension)
#rename_file_extensions(folder3, old_extension3, new_extension)